In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
from psilia import get_console
from psilia.data.mcap import (
    McapTaker as Taker, 
    get_mcap_overview
)
from psilia.data.utils import (
    psi_glob, 
    load_yaml, 
    save_yaml,
)
from psilia.vision.camera import (
    CameraIntrinsics, 
    unproject, 
    project,
)
from psilia.transforms import (
    Transform, 
    TransformTree,
    CAM_ALONG_X
)
from psilia.plotting import RerunLogger
import jax
import jax.numpy as jnp
import cv2
import numpy as np
import matplotlib.pyplot as plt
console = get_console()

In [ ]:
from psilia.plotting import RerunLogger

rrl = RerunLogger("Depth-Baselines")

In [ ]:
console = get_console()
print = console.print

path = str(Path('~/workspace/data/rosbags/**/*.mcap').expanduser())
mcaps = psi_glob(path)
console.print(mcaps[["name", "path"]])

mcap = mcaps["path"][1]
overview = get_mcap_overview(mcap)
console.print(overview)

In [ ]:
taker = Taker(mcap, 
    topic_map={
        "/zed/zed_node/pose": "/pose", 
        "/zed/zed_node/left/camera_info": "/left_intr",
        "/zed/zed_node/left/image_rect_color": "/left_im",
        "/zed/zed_node/right/camera_info": "/right_intr",
        "/zed/zed_node/right/image_rect_color": "/right_im",
        "/zed/zed_node/depth/depth_registered": "/depth",
        "/zed/zed_node/confidence/confidence_map": "/conf"
    }, 
    schema_transforms = {
        "sensor_msgs/msg/CameraInfo": {
            "__node__": lambda d: CameraIntrinsics.from_camera_info_dict(d),
        },
        "sensor_msgs/msg/Image": {
            "__node__": lambda d: d["data"]
        },
        "geometry_msgs/msg/PoseStamped" : {
            "__node__": lambda d: Transform.from_dict(d),
        }
    },
    topic_transforms={
        "/tf_static": {"__node__": lambda d: TransformTree(d["transforms"], strict=False)},
        "/conf": {"__node__": lambda d: 1.-d["data"][:,:,None]/100},
    })

In [ ]:
intr, tf_static = taker.k[
    "/left_intr", "/tf_static"].i[0,0,0].t(0, sort_key="publish_time")

console.print(intr)
console.print(tf_static)

In [ ]:
tf01 = tf_static["zed_left_camera_optical_frame", "zed_right_camera_optical_frame"]
B = tf01.t[0] # Baseline for depth from disparity
console.print(tf10)

In [ ]:
from psilia.vision.depth_cv import stereo_depth, _lr_depth


t = 5.0
im0, im1 = taker.k["/left_im", "/right_im"].i[0,0].t(t)
im0 = im0[...,:3]
im1 = im1[...,:3]
console.inspect(im0=im0, im1=im1)


depth = stereo_depth(
    np.array(im0),
    np.array(im1),
    intr,
    B,
    num_disparities = 64,
    block_size = 5,
    min_disparity = 0,
) 

console.inspect(depth=depth, conf=conf)

fig, axs = plt.subplots(1,2, figsize=(10,5))
axs[0].imshow(im0)
axs[1].imshow(depth, vmin=0.2, vmax=3)

In [ ]:
xs = unproject(depth[...,None], intr)
xs = CAM_ALONG_X(xs)

rrl.set_time(0)
rrl.log_points("Inferred_OPENCV", xs, cs=im0.reshape(-1,3), radii=0.0025)

In [ ]:
depth_zed, conf_zed = taker.k["/depth", "/conf"].i[0,0].t(t)

console.inspect(depth_zed=depth_zed, conf_zed=conf_zed)
xs = unproject(depth_zed[...,None], intr)
xs = CAM_ALONG_X(xs)

rrl.set_time(0)
rrl.log_points("Inferred_ZED", xs, cs=plt.get_cmap("viridis")(conf_zed.reshape(-1))[...,:3], radii=0.0025)

In [ ]:
depth_left, depth_right = _lr_depth(
    im0, 
    im1, 
    intr, 
    B,     
    num_disparities = 64,
    block_size = 5,
    min_disparity = 10
)

console.inspect(depth_left=depth_left, depth_right=depth_right)

xs_left = unproject(depth_left[...,None], intr)


xs_right = unproject(depth_right[...,None], intr)
xs_right = tf01(xs_right)

console.inspect(xs_right=xs_right, xs_left=xs_left)


fig, axs = plt.subplots(1,3, figsize=(10,5))
axs[0].imshow(im0)
axs[1].imshow(depth_left, vmin=0.2, vmax=3)
axs[2].imshow(depth_right, vmin=0.2, vmax=3)

In [ ]:
xs = unproject(depth_left[...,None], intr)
xs = CAM_ALONG_X(xs)

rrl.set_time(0)
rrl.log_points("Inferred", xs, cs=im0.reshape(-1,3), radii=0.0025)

In [ ]:
xs = unproject(depth_right[...,None], intr)
xs = CAM_ALONG_X(tf01(xs))

rrl.set_time(1)
rrl.log_points("Inferred", xs, cs=im1.reshape(-1,3), radii=0.0025)